In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import shap
import lime
import lime.lime_tabular
from tqdm.notebook import tqdm
from scipy.stats import pearsonr
from IPython.display import display

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import BayesianRidge, RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

shap.initjs()

TARGETS = {
    'ESG Score': 'corpgov_tresgscore',
    'Environment (E) Score': 'corpgov_environmentpillarscore',
    'Social (S) Score': 'corpgov_socialpillarscore',
    'Governance (G) Score': 'corpgov_governancepillarscore'
}

MAX_FEATURES_TO_TEST = 50
FEATURE_SEARCH_STEP = 5

def load_and_clean_data(filepath):
    print(f"--- Loading Data from {filepath} ---")
    try:
        df = pd.read_csv(filepath, low_memory=False)
    except FileNotFoundError:
        print("Error: File not found.")
        return None, None

    df.columns = ["".join(c if c.isalnum() else '_' for c in str(x)) for x in df.columns]

    leaky_keywords = ['score', 'grade', 'percentile', 'controversies', 'INDEXESG1', 'rank']
    potential_leaks = [c for c in df.columns if any(k in c.lower() for k in leaky_keywords)]

    return df, potential_leaks

def get_ensemble_ranking(X, y):
    s = RobustScaler()
    X_s = s.fit_transform(X)

    xgb = XGBRegressor(n_estimators=100, n_jobs=-1, random_state=42).fit(X_s, y)
    lgb = LGBMRegressor(n_estimators=100, verbosity=-1, random_state=42).fit(X_s, y)
    cat = CatBoostRegressor(iterations=100, verbose=0, random_state=42).fit(X, y)

    r1 = pd.Series(xgb.feature_importances_, index=X.columns).rank(pct=True)
    r2 = pd.Series(lgb.feature_importances_, index=X.columns).rank(pct=True)
    r3 = pd.Series(cat.feature_importances_, index=X.columns).rank(pct=True)

    consensus = (r1 + r2 + r3) / 3
    return consensus.sort_values(ascending=False).index.tolist()

df_full, potential_leaks = load_and_clean_data('/kaggle/input/datasets/tofael08.csv')

if df_full is not None:
    for target_nice_name, target_col in TARGETS.items():
        print(f"\n{'='*60}")
        print(f"PROCESSING TARGET: {target_nice_name}")
        print(f"{'='*60}")

        y = df_full[[target_col]].fillna(df_full[target_col].median())
        y_ravel = y.values.ravel()

        X = df_full.drop(columns=potential_leaks, errors='ignore')

        cols_drop_meta = [c for c in X.columns if ('iden' in c.lower()) or ('year' in c.lower())]
        X = X.drop(columns=cols_drop_meta)

        miss = X.isnull().mean()
        X = X.drop(columns=miss[miss > 0.4].index)

        X_train, X_test, y_train, y_test = train_test_split(X, y_ravel, test_size=0.2, random_state=42)

        cols = list(set(X_train.columns) & set(X_test.columns))
        X_train, X_test = X_train[cols], X_test[cols]

        num_cols = X_train.select_dtypes(include=np.number).columns
        cat_cols = X_train.select_dtypes(exclude=np.number).columns

        imp = SimpleImputer(strategy='median')
        X_train[num_cols] = imp.fit_transform(X_train[num_cols])
        X_test[num_cols] = imp.transform(X_test[num_cols])

        for c in cat_cols:
            X_train[c] = X_train[c].astype(str).fillna("Missing")
            X_test[c] = X_test[c].astype(str).fillna("Missing")
            le = LabelEncoder()
            le.fit(pd.concat([X_train[c], X_test[c]]))
            X_train[c] = le.transform(X_train[c])
            X_test[c] = le.transform(X_test[c])

        print(f"   > Features after leak removal: {X_train.shape[1]}")

        print("   > Running Ensemble Consensus Feature Ranking...")
        ranked_feats = get_ensemble_ranking(X_train, y_train)

        print("   > Optimizing feature count using 5-Fold CV (Training Set)...")
        best_rmse = float('inf')
        best_k = 10

        eval_model = StackingRegressor([('xgb', XGBRegressor(n_estimators=50, n_jobs=-1))], final_estimator=BayesianRidge())

        results_opt = []
        upper_lim = min(MAX_FEATURES_TO_TEST, len(ranked_feats))

        kf = KFold(n_splits=5, shuffle=True, random_state=42)

        for k in range(5, upper_lim + 1, FEATURE_SEARCH_STEP):
            tk = ranked_feats[:k]

            s = StandardScaler()
            Xt_s = s.fit_transform(X_train[tk])

            cv_scores = cross_val_score(eval_model, Xt_s, y_train,
                                        scoring='neg_mean_squared_error',
                                        cv=kf, n_jobs=-1)

            avg_rmse = np.sqrt(-cv_scores.mean())
            results_opt.append({'k': k, 'RMSE': avg_rmse})

            if avg_rmse < best_rmse:
                best_rmse = avg_rmse
                best_k = k

        print(f"   > Best k found (CV): {best_k} (CV RMSE: {best_rmse:.4f})")

        final_feats = ranked_feats[:best_k]
        X_tr_fin = X_train[final_feats]
        X_te_fin = X_test[final_feats]

        scaler = StandardScaler()
        X_tr_fin_s = scaler.fit_transform(X_tr_fin)
        X_te_fin_s = scaler.transform(X_te_fin)

        print("   > Training Final Models (Base + 2 Meta Learners)...")

        base_models = [
            ('XGB', XGBRegressor(n_estimators=200, max_depth=5, random_state=42)),
            ('LGBM', LGBMRegressor(n_estimators=200, verbosity=-1, random_state=42)),
            ('CAT', CatBoostRegressor(iterations=200, depth=6, verbose=0, random_state=42)),
            ('GBOOST', GradientBoostingRegressor(n_estimators=200, random_state=42))
        ]

        meta_bayes = StackingRegressor(estimators=base_models, final_estimator=BayesianRidge(), n_jobs=-1)
        meta_svr = StackingRegressor(estimators=base_models, final_estimator=SVR(C=1.0, epsilon=0.1), n_jobs=-1)

        models = {
            'XGB': base_models[0][1],
            'LGBM': base_models[1][1],
            'CAT': base_models[2][1],
            'GBOOST': base_models[3][1],
            'Stack_Bayes': meta_bayes,
            'Stack_SVR': meta_svr
        }

        res_list = []
        preds_dict = {}

        for name, model in models.items():
            model.fit(X_tr_fin_s, y_train)
            preds = model.predict(X_te_fin_s)
            preds_dict[name] = preds

            rmse = np.sqrt(mean_squared_error(y_test, preds))
            mae = mean_absolute_error(y_test, preds)
            r2 = r2_score(y_test, preds)
            res_list.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})

        res_df = pd.DataFrame(res_list).set_index('Model').sort_values('RMSE')
        print(f"\n   --- Results for {target_nice_name} (Held-out Test Set) ---")
        print(res_df)

        best_model_name = res_df.index[0]
        best_preds = preds_dict[best_model_name]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        ax1.scatter(y_test, best_preds, alpha=0.5, color='#2A9D8F')
        ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        ax1.set_title(f"True vs Predicted: {target_nice_name} ({best_model_name})")
        ax1.set_xlabel("True Score")
        ax1.set_ylabel("Predicted Score")
        ax1.grid(True)

        residuals = y_test - best_preds
        sns.boxplot(y=residuals, ax=ax2, color='#E9C46A')
        ax2.axhline(0, color='red', linestyle='--')
        ax2.set_title(f"Residual Distribution ({target_nice_name})")
        ax2.set_ylabel("Error (True - Pred)")

        plt.tight_layout()
        plt.show()

        print(f"\n   > Generating XAI for {target_nice_name}...")

        try:
            shap.initjs()

            X_bg = shap.sample(X_tr_fin_s, 50, random_state=42)
            X_exp = X_te_fin_s[:50]

            explainer = shap.KernelExplainer(models[best_model_name].predict, X_bg)
            shap_vals = explainer.shap_values(X_exp)

            exp_val = explainer.expected_value
            if isinstance(exp_val, (list, np.ndarray)):
                exp_val = exp_val[0]

            plt.figure(figsize=(10, 5))
            shap.summary_plot(shap_vals, X_exp, feature_names=final_feats, show=False)
            plt.title(f"SHAP Summary: {target_nice_name}")
            plt.tight_layout()
            plt.show()

            print(f"     > Generating SHAP Interactive Force Plots...")

            subset_preds = models[best_model_name].predict(X_exp)
            high_idx = np.argmax(subset_preds)
            low_idx = np.argmin(subset_preds)

            print(f"\n       Force Plot A: Highest Prediction in Sample ({subset_preds[high_idx]:.2f})")
            force_high = shap.force_plot(exp_val, shap_vals[high_idx], X_exp[high_idx], feature_names=final_feats)
            display(force_high)

            print(f"\n       Force Plot B: Lowest Prediction in Sample ({subset_preds[low_idx]:.2f})")
            force_low = shap.force_plot(exp_val, shap_vals[low_idx], X_exp[low_idx], feature_names=final_feats)
            display(force_low)

            print(f"\n       Force Plot C: Stacked view of all 50 explained samples (Order by Similarity)")
            force_stacked = shap.force_plot(exp_val, shap_vals, X_exp, feature_names=final_feats)
            display(force_stacked)

        except Exception as e:
            print(f"SHAP Error (skipping): {e}")

        try:
            lime_exp = lime.lime_tabular.LimeTabularExplainer(
                training_data=X_tr_fin_s,
                feature_names=final_feats,
                mode='regression',
                verbose=False
            )

            high_idx = np.argmax(best_preds)
            print(f"     LIME: Explaining Sample with HIGHEST Pred ({best_preds[high_idx]:.2f})")
            exp = lime_exp.explain_instance(X_te_fin_s[high_idx], models[best_model_name].predict, num_features=8)
            exp.as_pyplot_figure()
            plt.title(f"LIME High Score Scenario: {target_nice_name}")
            plt.show()

        except Exception as e:
            print(f"LIME Error (skipping): {e}")

    print("\n--- ALL TARGETS PROCESSED SUCCESSFULLY ---")